In [ ]:
"""
CLARANS clustering with GPU-accelerated preprocessing (CuPy/cuML when available) and CPU CLARANS.
Replicates the main functionality of the updated notebook sections (Clarans predefined and Bayesian Search).

Outputs:
- Cluster visualizations, elbow plots, quality analyses, comparison tables and plots saved to CWD.

Notes:
- CLARANS remains CPU via pyclustering.
- StandardScaler and PCA use cuML on GPU when available, then data are converted back to NumPy before CLARANS.
- Safe fallback to CPU sklearn when GPU libraries are unavailable.
"""

import os
import time
from datetime import datetime
from pathlib import Path
from collections import Counter
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)
from sklearn.preprocessing import StandardScaler as SkStandardScaler
from sklearn.decomposition import PCA as SkPCA

from scipy.spatial.distance import pdist

from pyclustering.cluster.clarans import clarans

warnings.filterwarnings("ignore")

# GPU setup (CuPy/cuML) with CPU fallback
GPU_AVAILABLE = False
try:
    import cupy as cp  # type: ignore
    from cuml.decomposition import PCA as CuPCA  # type: ignore
    from cuml.preprocessing import StandardScaler as CuStandardScaler  # type: ignore
    GPU_AVAILABLE = True
    print("GPU available: using CuPy/cuML where possible.")
except Exception as e:
    print("cuML/CuPy not available; running on CPU. Reason:", e)


def resolve_code_dir() -> Path:
    # script location: .../Code/Machine_Learning/UnSupervised_Models/CLARANS/predefined
    here = Path(__file__).resolve().parent
    code_dir = here.parent.parent.parent.parent  # Go up to Code directory
    return code_dir


def get_paths(code_dir: Path) -> dict:
    return {
        'X_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote.parquet',
        'X_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_tomek.parquet',
        'X_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote_tomek.parquet',
        'y_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote' / 'y_smote.pkl',
        'y_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'tomek' / 'y_tomek.pkl',
        'y_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote_tomek' / 'y_smote_tomek.pkl',
        'X_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'X_val.parquet',
        'X_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'X_test.parquet',
        'y_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'y_val.pkl',
        'y_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'y_test.pkl',
    }


def load_all_data(paths: dict) -> dict:
    data = {}
    print("\n" + "="*50)
    print("LOADING DATA")
    print("="*50)

    parquet_keys = ['X_train_smote', 'X_train_tomek', 'X_train_smote_tomek', 'X_val', 'X_test']
    for key in parquet_keys:
        path = paths[key]
        if path.exists():
            try:
                data[key] = pd.read_parquet(path)
                print(f"✓ Loaded {key}: {data[key].shape}")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")

    pickle_keys = ['y_train_smote', 'y_train_tomek', 'y_train_smote_tomek', 'y_val', 'y_test']
    for key in pickle_keys:
        path = paths[key]
        if path.exists():
            try:
                import pickle
                with open(path, 'rb') as f:
                    data[key] = pickle.load(f)
                print(f"✓ Loaded {key}: {len(data[key])} samples")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")

    return data


# ---------- Utility functions ----------

def calculate_clustering_metrics(X: np.ndarray, labels: np.ndarray, method_name: str) -> dict:
    if len(np.unique(labels)) < 2:
        return {
            'silhouette': 0,
            'calinski_harabasz': 0,
            'davies_bouldin': float('inf'),
            'n_clusters': len(np.unique(labels)),
            'cluster_sizes': Counter(labels),
            'method': method_name,
        }
    try:
        silhouette = silhouette_score(X, labels)
    except Exception:
        silhouette = 0
    try:
        calinski = calinski_harabasz_score(X, labels)
    except Exception:
        calinski = 0
    try:
        davies = davies_bouldin_score(X, labels)
    except Exception:
        davies = float('inf')
    return {
        'silhouette': silhouette,
        'calinski_harabasz': calinski,
        'davies_bouldin': davies,
        'n_clusters': len(np.unique(labels)),
        'cluster_sizes': Counter(labels),
        'method': method_name,
    }


def plot_clusters_2d(X: np.ndarray, labels: np.ndarray, centers: np.ndarray, method_name: str, timestamp: str) -> str:
    if X.shape[1] > 2:
        pca = SkPCA(n_components=2, random_state=0)
        X_2d = pca.fit_transform(X)
        centers_2d = pca.transform(centers) if centers is not None else None
        variance_exp = pca.explained_variance_ratio_
        title_suffix = f" (PCA: {variance_exp[0]:.1%}+{variance_exp[1]:.1%} variance)"
    else:
        X_2d = X
        centers_2d = centers
        title_suffix = ""

    plt.figure(figsize=(12, 10))
    unique_labels = np.unique(labels)
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
    for label, color in zip(unique_labels, colors):
        mask = labels == label
        plt.scatter(X_2d[mask, 0], X_2d[mask, 1], c=[color], label=f'Cluster {label}',
                    alpha=0.6, s=50, edgecolors='w', linewidth=0.5)
    if centers_2d is not None:
        plt.scatter(centers_2d[:, 0], centers_2d[:, 1], c='red', marker='X', s=300,
                    label='Centroids', edgecolors='black', linewidth=2)
    plt.title(f'Clusters CLARANS - {method_name}{title_suffix}', fontsize=16, fontweight='bold')
    plt.xlabel('Component 1', fontsize=12)
    plt.ylabel('Component 2', fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    filename = f"clarans_clusters_{method_name.lower().replace(' ', '_')}_{timestamp}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    return filename


def plot_elbow_method(inertias, k_values, method_name, timestamp) -> str:
    plt.figure(figsize=(10, 6))
    plt.plot(k_values, inertias, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Nombre de clusters (k)', fontsize=12)
    plt.ylabel('Inertie (Within-cluster SSE)', fontsize=12)
    plt.title(f'Méthode Elbow - CLARANS - {method_name}', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    if len(inertias) >= 3:
        second_diff = np.diff(np.diff(inertias))
        if len(second_diff) > 0:
            elbow_idx = int(np.argmax(second_diff)) + 1
            if elbow_idx < len(k_values):
                plt.axvline(x=k_values[elbow_idx], color='r', linestyle='--', alpha=0.7,
                            label=f'Suggested k = {k_values[elbow_idx]}')
                plt.legend()
    filename = f"clarans_elbow_{method_name.lower().replace(' ', '_')}_{timestamp}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    return filename


def analyze_cluster_quality(X: np.ndarray, labels: np.ndarray, method_name: str, timestamp: str):
    n_clusters = len(np.unique(labels))
    intra_distances = []
    inter_distances = []
    for i in range(n_clusters):
        cluster_points = X[labels == i]
        if len(cluster_points) > 1:
            intra_dist = np.mean(pdist(cluster_points))
            intra_distances.append(intra_dist)
        for j in range(i + 1, n_clusters):
            other_points = X[labels == j]
            if len(other_points) > 0:
                dist_matrix = np.sqrt(((cluster_points[:, np.newaxis] - other_points) ** 2).sum(axis=2))
                inter_distances.append(dist_matrix.min())
    metrics = {
        'n_clusters': n_clusters,
        'avg_intra_distance': np.mean(intra_distances) if intra_distances else 0,
        'min_inter_distance': np.min(inter_distances) if inter_distances else 0,
        'separation_ratio': (np.min(inter_distances) / np.mean(intra_distances)) if intra_distances and inter_distances else 0,
        'method': method_name,
    }

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    cluster_sizes = [np.sum(labels == i) for i in range(n_clusters)]
    axes[0].bar(range(n_clusters), cluster_sizes, color='skyblue', edgecolor='black')
    axes[0].set_xlabel('Cluster ID', fontsize=12)
    axes[0].set_ylabel('Nombre de points', fontsize=12)
    axes[0].set_title(f'Distribution des tailles de clusters\n{method_name}', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    for i, size in enumerate(cluster_sizes):
        axes[0].text(i, size + max(cluster_sizes)*0.01, str(size), ha='center', va='bottom', fontsize=10)
    if intra_distances:
        axes[1].boxplot(intra_distances, patch_artist=True, boxprops=dict(facecolor='lightgreen'))
        axes[1].set_xlabel('Clusters', fontsize=12)
        axes[1].set_ylabel('Distance intra-cluster moyenne', fontsize=12)
        axes[1].set_title('Distribution des distances intra-cluster', fontsize=14)
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, "Pas assez de données\npour l'analyse", ha='center', va='center', fontsize=12)
    plt.suptitle(f'Analyse de la qualité des clusters - {method_name}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    filename = f"clarans_quality_{method_name.lower().replace(' ', '_')}_{timestamp}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    return metrics, filename


# ---------- CLARANS runners ----------

def run_clarans_clustering(X: np.ndarray, n_clusters: int, method_name: str, timestamp: str,
                           num_local: int = 2, max_neighbors: int = 50) -> dict:
    print(f"\n🔍 Exécution de CLARANS pour {n_clusters} clusters...")
    start_time = time.time()

    data_points = X.tolist()
    clarans_instance = clarans(data_points, n_clusters, num_local, max_neighbors)
    clarans_instance.process()

    clusters = clarans_instance.get_clusters()
    medoids = clarans_instance.get_medoids()

    labels = np.zeros(len(X), dtype=int)
    for cluster_id, cluster in enumerate(clusters):
        for point_idx in cluster:
            labels[point_idx] = cluster_id

    centers = X[medoids]
    elapsed_time = time.time() - start_time
    print(f"✅ CLARANS terminé en {elapsed_time:.2f} secondes")

    metrics = calculate_clustering_metrics(X, labels, method_name)
    metrics['execution_time'] = elapsed_time

    cluster_plot = plot_clusters_2d(X, labels, centers, method_name, timestamp)
    quality_metrics, quality_plot = analyze_cluster_quality(X, labels, method_name, timestamp)
    metrics.update(quality_metrics)

    return {
        'labels': labels,
        'centers': centers,
        'medoids': medoids,
        'clusters': clusters,
        'metrics': metrics,
        'plots': {
            'clusters': cluster_plot,
            'quality': quality_plot,
        }
    }


def find_optimal_k_elbow(X: np.ndarray, method_name: str, timestamp: str, k_range=range(2, 11)):
    print(f"\n📊 Recherche du nombre optimal de clusters (k) avec méthode elbow...")
    inertias = []
    results_by_k = {}
    data_points = X.tolist()

    for k in k_range:
        print(f"  Test avec k={k}...", end=' ')
        start_time = time.time()

        clarans_instance = clarans(data_points, k, 2, 20)
        clarans_instance.process()

        clusters = clarans_instance.get_clusters()
        medoids = clarans_instance.get_medoids()

        inertia = 0.0
        for cluster_id, cluster in enumerate(clusters):
            medoid = data_points[medoids[cluster_id]]
            for point_idx in cluster:
                point = data_points[point_idx]
                dist = sum((a - b) ** 2 for a, b in zip(point, medoid))
                inertia += dist

        inertias.append(inertia)
        elapsed_time = time.time() - start_time
        print(f"inertie={inertia:.2f}, temps={elapsed_time:.1f}s")

        labels = np.zeros(len(X), dtype=int)
        for cluster_id, cluster in enumerate(clusters):
            for point_idx in cluster:
                labels[point_idx] = cluster_id

        results_by_k[k] = {
            'inertia': inertia,
            'labels': labels,
            'medoids': medoids,
        }

    elbow_plot = plot_elbow_method(inertias, list(k_range), method_name, timestamp)

    if len(inertias) >= 3:
        second_diff = np.diff(np.diff(inertias))
        if len(second_diff) > 0:
            optimal_k_idx = int(np.argmax(second_diff)) + 1
            optimal_k = list(k_range)[optimal_k_idx]
        else:
            optimal_k = 3
    else:
        optimal_k = 3

    print(f"✅ k optimal suggéré: {optimal_k}")
    return optimal_k, inertias, elbow_plot, results_by_k


# ---------- Main workflow ----------

def main():
    code_dir = resolve_code_dir()
    print(f"Code directory: {code_dir}")

    paths = get_paths(code_dir)
    print("\nData paths:")
    for key, path in paths.items():
        exists = "✓" if path.exists() else "✗"
        print(f"  {exists} {key}: {path}")

    data = load_all_data(paths)
    if not data:
        print("No data was loaded. Exiting.")
        return

    print(f"\n{'='*80}")
    print("CLARANS CLUSTERING - ÉVALUATION SUR TOUTES LES DONNÉES RESAMPLÉES")
    print(f"{'='*80}")
    print(f"✅ Données chargées: {len(data)} datasets")

    METHODS = [
        ('SMOTE', 'X_train_smote', 'y_train_smote'),
        ('Tomek Links', 'X_train_tomek', 'y_train_tomek'),
        ('SMOTE + Tomek Links', 'X_train_smote_tomek', 'y_train_smote_tomek'),
    ]

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    all_results = []

    for method_name, X_train_key, y_train_key in METHODS:
        print(f"\n{'='*80}")
        print(f"🚀 CLARANS CLUSTERING - {method_name}")
        print(f"{'='*80}")

        if X_train_key not in data or y_train_key not in data:
            print(f"⚠️  Données {method_name} non disponibles, skip...")
            continue

        X_train = data[X_train_key].values
        y_train = data[y_train_key]

        print(f"📊 Données: {X_train.shape[0]} échantillons, {X_train.shape[1]} features")
        print(f"🎯 Distribution des classes originales: {Counter(y_train)}")

        # GPU-accelerated PCA (if many features), then back to CPU for CLARANS
        if X_train.shape[1] > 50:
            print(f"⚠️  Trop de features ({X_train.shape[1]}), application de PCA...")
            if GPU_AVAILABLE:
                print("   Using GPU PCA (cuML)...")
                pca = CuPCA(n_components=50, random_state=0)
                X_gpu = cp.asarray(X_train)
                X_train_gpu = pca.fit_transform(X_gpu)
                X_train = cp.asnumpy(X_train_gpu)
                var_sum = float(cp.asnumpy(pca.explained_variance_ratio_).sum())
            else:
                pca = SkPCA(n_components=50, random_state=0)
                X_train = pca.fit_transform(X_train)
                var_sum = float(pca.explained_variance_ratio_.sum())
            print(f"✅ Réduction à {X_train.shape[1]} composantes principales")
            print(f"   Variance expliquée: {var_sum:.2%}")

        # Step 1: optimal k via elbow
        print("\n" + "-" * 50)
        print("ÉTAPE 1: DÉTERMINATION DU NOMBRE OPTIMAL DE CLUSTERS")
        print("-" * 50)
        optimal_k, inertias, elbow_plot, k_results = find_optimal_k_elbow(
            X_train, method_name, timestamp, k_range=range(2, 11)
        )

        # Step 2: run CLARANS with optimal k
        print("\n" + "-" * 50)
        print(f"ÉTAPE 2: CLARANS AVEC k={optimal_k}")
        print("-" * 50)
        clarans_results = run_clarans_clustering(
            X_train, optimal_k, method_name, timestamp, num_local=5, max_neighbors=100
        )

        # Step 3: compare with original labels
        print("\n" + "-" * 50)
        print("ÉTAPE 3: COMPARAISON AVEC LES LABELS ORIGINAUX")
        print("-" * 50)
        cluster_labels = clarans_results['labels']
        if y_train is not None:
            ari = adjusted_rand_score(y_train, cluster_labels)
            nmi = normalized_mutual_info_score(y_train, cluster_labels)
            print("📊 Comparaison clusters vs labels originaux:")
            print(f"   Adjusted Rand Index: {ari:.4f}")
            print(f"   Normalized Mutual Info: {nmi:.4f}")

            confusion = pd.crosstab(pd.Series(cluster_labels, name='Cluster'),
                                     pd.Series(y_train, name='Classe réelle'))
            plt.figure(figsize=(10, 8))
            sns.heatmap(confusion, annot=True, fmt='d', cmap='YlOrRd')
            plt.title(f'Clusters vs Classes réelles - {method_name}', fontsize=14, fontweight='bold')
            plt.tight_layout()
            confusion_filename = f"clarans_confusion_{method_name.lower().replace(' ', '_')}_{timestamp}.png"
            plt.savefig(confusion_filename, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"💾 Matrice de confusion sauvegardée: {confusion_filename}")
            clarans_results['comparison_metrics'] = {
                'adjusted_rand_index': ari,
                'normalized_mutual_info': nmi,
            }
            clarans_results['plots']['confusion'] = confusion_filename

        method_result = {
            'method': method_name,
            'data_shape': X_train.shape,
            'optimal_k': optimal_k,
            'clarans_results': clarans_results,
            'elbow_plot': elbow_plot,
            'k_results': k_results,
        }
        all_results.append(method_result)

        print("\n📋 RÉSUMÉ DES RÉSULTATS:")
        print("-" * 50)
        metrics = clarans_results['metrics']
        print(f"Nombre de clusters: {metrics['n_clusters']}")
        print(f"Taille des clusters: {dict(metrics['cluster_sizes'])}")
        print(f"Silhouette Score: {metrics['silhouette']:.4f}")
        print(f"Calinski-Harabasz Index: {metrics['calinski_harabasz']:.2f}")
        print(f"Davies-Bouldin Index: {metrics['davies_bouldin']:.4f}")
        print(f"Temps d'exécution: {metrics['execution_time']:.2f}s")
        print(f"Distance intra-cluster moyenne: {metrics['avg_intra_distance']:.4f}")
        print(f"Distance inter-cluster minimale: {metrics['min_inter_distance']:.4f}")
        print(f"Ratio de séparation: {metrics['separation_ratio']:.4f}")
        if 'comparison_metrics' in clarans_results:
            print(f"Adjusted Rand Index: {clarans_results['comparison_metrics']['adjusted_rand_index']:.4f}")

        print("\n💾 Fichiers générés:")
        for plot_name, plot_file in clarans_results['plots'].items():
            print(f"  • {plot_name}: {plot_file}")

    # Final comparison across methods
    if all_results:
        print("\n" + "=" * 80)
        print("📊 COMPARAISON FINALE DES MÉTHODES")
        print("=" * 80)
        comparison_data = []
        for result in all_results:
            metrics = result['clarans_results']['metrics']
            comp_data = {
                'Méthode': result['method'],
                'Échantillons': result['data_shape'][0],
                'Features': result['data_shape'][1],
                'k optimal': result['optimal_k'],
                'Silhouette': metrics['silhouette'],
                'Calinski-Harabasz': metrics['calinski_harabasz'],
                'Davies-Bouldin': metrics['davies_bouldin'],
                'Temps (s)': metrics['execution_time'],
                'Distance intra': metrics['avg_intra_distance'],
                'Distance inter': metrics['min_inter_distance'],
                'Ratio séparation': metrics['separation_ratio'],
            }
            if 'comparison_metrics' in result['clarans_results']:
                comp_data['ARI'] = result['clarans_results']['comparison_metrics']['adjusted_rand_index']
                comp_data['NMI'] = result['clarans_results']['comparison_metrics']['normalized_mutual_info']
            comparison_data.append(comp_data)

        comparison_df = pd.DataFrame(comparison_data)
        print("\n📋 TABLEAU COMPARATIF:")
        print("-" * 120)
        print(comparison_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
        print("-" * 120)
        comparison_filename = f"clarans_comparison_{timestamp}.csv"
        comparison_df.to_csv(comparison_filename, index=False)
        print(f"\n💾 Tableau comparatif sauvegardé: {comparison_filename}")

        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        methods = [r['method'] for r in all_results]
        metrics_to_plot = [
            ('Silhouette', 'silhouette', 'Score', 'higher_better'),
            ('Calinski-Harabasz', 'calinski_harabasz', 'Score', 'higher_better'),
            ('Davies-Bouldin', 'davies_bouldin', 'Score', 'lower_better'),
            ("Temps d'exécution", 'execution_time', 'Secondes', 'lower_better'),
            ('Distance intra-cluster', 'avg_intra_distance', 'Distance', 'lower_better'),
            ('Ratio de séparation', 'separation_ratio', 'Ratio', 'higher_better'),
        ]
        for idx, (title, metric_key, ylabel, better_type) in enumerate(metrics_to_plot):
            ax = axes[idx // 3, idx % 3]
            values = [r['clarans_results']['metrics'][metric_key] for r in all_results]
            bars = ax.bar(methods, values, color=['#1f77b4', '#ff7f0e', '#2ca02c'], edgecolor='black', alpha=0.8)
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.set_ylabel(ylabel, fontsize=10)
            ax.set_xticklabels(methods, rotation=45, ha='right')
            ax.grid(True, alpha=0.3)
            for bar in bars:
                height = bar.get_height()
                if metric_key == 'davies_bouldin' and height > 100:
                    ax.text(bar.get_x() + bar.get_width()/2., height, f"{height:.1e}", ha='center', va='bottom', fontsize=9)
                else:
                    ax.text(bar.get_x() + bar.get_width()/2., height, f"{height:.3f}", ha='center', va='bottom', fontsize=9)
            if better_type == 'higher_better':
                best_idx = int(np.argmax(values))
                bars[best_idx].set_color('green')
                bars[best_idx].set_edgecolor('darkgreen')
            elif better_type == 'lower_better':
                best_idx = int(np.argmin(values))
                bars[best_idx].set_color('red')
                bars[best_idx].set_edgecolor('darkred')
        plt.suptitle(f'Comparaison des méthodes - CLARANS Clustering\n{timestamp}', fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        comparison_plot_filename = f"clarans_comparison_plot_{timestamp}.png"
        plt.savefig(comparison_plot_filename, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"\n💾 Graphique de comparaison sauvegardé: {comparison_plot_filename}")

    print("\n" + "=" * 80)
    print("✅ ANALYSE CLARANS TERMINÉE AVEC SUCCÈS!")
    print("=" * 80)


if __name__ == "__main__":
    main()
